In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "mobilenet_v2",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}

train_dir = "../BanglaLekha_dataset_split/train"
val_dir = "../BanglaLekha_dataset_split/validation"
test_dir = "../BanglaLekha_dataset_split/test"


train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [6]:

mobilenet_v2 = models.mobilenet_v2(pretrained=True)


for param in mobilenet_v2.parameters():
    param.requires_grad = False

mobilenet_v2.classifier[1] = nn.Linear(mobilenet_v2.last_channel, 84)

total_params = sum(p.numel() for p in mobilenet_v2.parameters())

gpu = torch.device("cuda")
mobilenet_v2 = mobilenet_v2.to(gpu)


In [7]:
print(total_params)

2331476


In [8]:
import fine_tuning as ft
mobilenet_v2 = ft.fine_tune(model=mobilenet_v2, model_name="mobilenet_v2", state="full") #Change the state for fine tuning

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam([
    {"params": mobilenet_v2.classifier[1].parameters(), "lr": 1e-3},
    {"params": mobilenet_v2.features.parameters(), "lr": 1e-5},
    
])
epochs = model_config["epochs"]

In [10]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [11]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs):
    
    max_val_accuracy = 0
    count = 0
    patience = 5

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(mobilenet_v2, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), "saved_parameters/mobilenet_v2.pth")

        else:
            count = count + 1
        
        if(count >= patience):
            break

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [19]:
train_model(mobilenet_v2, train_dataloader, val_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.29490741844855245, Validation Accuracy: 0.9077748959527112
Epoch: 2, Training Loss: 0.257933444392479, Validation Accuracy: 0.9062066469630256
Epoch: 3, Training Loss: 0.23690690031186165, Validation Accuracy: 0.9134447192231135
Epoch: 4, Training Loss: 0.21791144289904554, Validation Accuracy: 0.9174859762349961
Epoch: 5, Training Loss: 0.19208763387680347, Validation Accuracy: 0.9144097955244587
Epoch: 6, Training Loss: 0.17998472708073532, Validation Accuracy: 0.9244827794197479
Epoch: 7, Training Loss: 0.16754498722911307, Validation Accuracy: 0.9185716870740093
Epoch: 8, Training Loss: 0.15158274572113564, Validation Accuracy: 0.9186923216116775
Epoch: 9, Training Loss: 0.1410749688805263, Validation Accuracy: 0.9238796067314072
Epoch: 10, Training Loss: 0.12900750075923514, Validation Accuracy: 0.9195970806441884


In [12]:
mobilenet_v2.load_state_dict(torch.load("saved_parameters/mobilenet_v2.pth"))
mobilenet_v2.eval()

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [14]:
accuracy = validate_model(mobilenet_v2, test_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 91.74201688245365
